# Synthetic Wake Word Mini-Dataset Generator
## NGI0 Commons Fund deliverable — WP2

This notebook generates a **20-sample synthetic wake-word mini-dataset** using
[edge-tts](https://github.com/rany2/edge-tts) (free, CPU-only, no API key).
The output is an LJSpeech-style directory ready to feed into the
[ww-trainer](https://github.com/OpenVoiceOS/ww-trainer) training pipeline.

**What you will learn:**
1. How to synthesise labelled wake-word audio samples from text using multiple TTS voices
2. How a synthetic dataset is structured (positive samples + metadata)
3. How to generate phonetically confusable adversarial negatives for evaluation

This is the CPU-feasible subset of the full pipeline in `tts2ww_full_pipeline.ipynb`
(which additionally handles voice conversion augmentation requiring a GPU).

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.

**CI execution:** ✅ executed headlessly with real audio outputs.


## 0 · Configuration

In [1]:
import nest_asyncio
nest_asyncio.apply()

import os

WAKE_WORD = "hey mycroft"        # phrase to synthesise
N_POSITIVE = 20                  # positive samples to generate
N_ADVERSARIAL = 10               # confusable negatives to generate (text only)
LANG = "en"

# edge-tts voice pool — use several voices for speaker diversity
VOICES = [
    "en-US-JennyNeural",
    "en-US-GuyNeural",
    "en-GB-SoniaNeural",
    "en-GB-RyanNeural",
    "en-AU-NatashaNeural",
]

print(f"Wake word : {WAKE_WORD!r}")
print(f"Positives : {N_POSITIVE}")
print(f"Voices    : {VOICES}")


Wake word : 'hey mycroft'
Positives : 20
Voices    : ['en-US-JennyNeural', 'en-US-GuyNeural', 'en-GB-SoniaNeural', 'en-GB-RyanNeural', 'en-AU-NatashaNeural']


## 1 · Install / verify edge-tts

In [2]:
import subprocess, sys

try:
    import edge_tts
    print(f"edge-tts already installed: {edge_tts.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "edge-tts"])
    import edge_tts
    print(f"edge-tts installed: {edge_tts.__version__}")


edge-tts already installed: 7.2.7


## 2 · Synthesise positive samples

We cycle through the voice pool to produce `N_POSITIVE` utterances of the
wake-word phrase.  Each file is 16 kHz mono WAV.


In [3]:
import asyncio, subprocess, tempfile
from pathlib import Path
from itertools import cycle

WORKDIR = Path(tempfile.mkdtemp(prefix="synth_ww_"))
POS_DIR = WORKDIR / "positives"
POS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {WORKDIR}")

async def synth(text: str, voice: str, out_mp3: Path):
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(out_mp3))

def mp3_to_wav(mp3: Path, wav: Path):
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(mp3), "-ar", "16000", "-ac", "1", str(wav)],
        check=True, capture_output=True,
    )

voice_cycle = cycle(VOICES)
generated = []

for i in range(N_POSITIVE):
    voice = next(voice_cycle)
    mp3 = POS_DIR / f"pos_{i:04d}.mp3"
    wav = POS_DIR / f"pos_{i:04d}.wav"
    asyncio.run(synth(WAKE_WORD, voice, mp3))
    mp3_to_wav(mp3, wav)
    mp3.unlink()  # keep only WAV
    size_kb = wav.stat().st_size // 1024
    generated.append({"file": wav.name, "voice": voice, "label": WAKE_WORD})
    print(f"  [{i+1:2d}/{N_POSITIVE}]  {wav.name}  {voice}  ({size_kb} KB)")

print(f"\n{len(generated)} positive samples generated.")


Output directory: /tmp/synth_ww_lgy813_9


  [ 1/20]  pos_0000.wav  en-US-JennyNeural  (60 KB)


  [ 2/20]  pos_0001.wav  en-US-GuyNeural  (57 KB)


  [ 3/20]  pos_0002.wav  en-GB-SoniaNeural  (48 KB)


  [ 4/20]  pos_0003.wav  en-GB-RyanNeural  (61 KB)


...

  [16/20]  pos_0015.wav  en-US-JennyNeural  (60 KB)


  [17/20]  pos_0016.wav  en-US-GuyNeural  (57 KB)


  [18/20]  pos_0017.wav  en-GB-SoniaNeural  (48 KB)


  [19/20]  pos_0018.wav  en-GB-RyanNeural  (61 KB)


  [20/20]  pos_0019.wav  en-AU-NatashaNeural  (61 KB)

20 positive samples generated.


## 3 · Write metadata CSV

The LJSpeech format uses pipe-separated `ID|normalised_text|text`.


In [4]:
import csv

metadata_path = WORKDIR / "metadata.csv"
with open(metadata_path, "w", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    writer.writerow(["id", "label", "voice"])
    for row in generated:
        stem = Path(row["file"]).stem
        writer.writerow([stem, row["label"], row["voice"]])

print(f"Metadata written: {metadata_path}")
print(f"Rows: {len(generated)}")

# Quick preview
with open(metadata_path) as f:
    for line in f.readlines()[:5]:
        print(" ", line.rstrip())


Metadata written: /tmp/synth_ww_lgy813_9/metadata.csv
Rows: 20
  id|label|voice
  pos_0000|hey mycroft|en-US-JennyNeural
  pos_0001|hey mycroft|en-US-GuyNeural
  pos_0002|hey mycroft|en-GB-SoniaNeural
  pos_0003|hey mycroft|en-GB-RyanNeural


## 4 · Generate adversarial negatives (text only, CPU)

Adversarial negatives are phonetically confusable words/phrases.
We use a simple grapheme-edit augmenter (single insertion/deletion/substitution)
which is the same method used in the full `tts2ww` pipeline.

> **Note:** Synthesising audio for negatives follows the same pattern as positives.
> Omitted here for brevity — the text file is the input to the ww-trainer
> negative-mining step.


In [5]:
import random
import string

random.seed(42)

def grapheme_edits(word: str, n: int = 10) -> list:
    results = set()
    chars = list(word)
    # substitutions
    for i in range(len(chars)):
        for c in "bcdfghjklmnpqrstvwxyz":
            candidate = chars[:i] + [c] + chars[i+1:]
            results.add("".join(candidate))
    # insertions
    for i in range(len(chars) + 1):
        for c in "aeiou":
            candidate = chars[:i] + [c] + chars[i:]
            results.add("".join(candidate))
    # deletions
    for i in range(len(chars)):
        candidate = chars[:i] + chars[i+1:]
        if len(candidate) > 2:
            results.add("".join(candidate))
    # filter: remove original and empties
    results.discard(word)
    results = [r for r in results if r.strip()]
    return random.sample(list(results), min(n, len(results)))

# Generate for each word in the wake word
all_adversarials = set()
for token in WAKE_WORD.split():
    edits = grapheme_edits(token, n=N_ADVERSARIAL // len(WAKE_WORD.split()))
    all_adversarials.update(edits)

# Save to text file
adv_path = WORKDIR / f"{WAKE_WORD.replace(' ', '_')}_adversarials.txt"
adv_list = sorted(all_adversarials)[:N_ADVERSARIAL]
adv_path.write_text("\n".join(adv_list) + "\n")

print(f"Adversarial negatives ({len(adv_list)}):")
for w in adv_list:
    print(f"  {w}")
print(f"\nSaved to: {adv_path}")


Adversarial negatives (10):
  dycroft
  hec
  hek
  hep
  heuy
  heyi
  mycboft
  mycraoft
  mycrcft
  mycrqft

Saved to: /tmp/synth_ww_lgy813_9/hey_mycroft_adversarials.txt


## 5 · Dataset summary

In [6]:
from pathlib import Path
import os

wavs = sorted(POS_DIR.glob("*.wav"))
total_size_kb = sum(w.stat().st_size for w in wavs) // 1024
avg_size_kb = total_size_kb // len(wavs) if wavs else 0

print("=== Synthetic wake-word mini-dataset ===")
print(f"Wake word      : {WAKE_WORD!r}")
print(f"Positive WAVs  : {len(wavs)}")
print(f"Total size     : {total_size_kb} KB")
print(f"Avg per sample : {avg_size_kb} KB")
print(f"Voices used    : {len(VOICES)}")
print(f"Adversarials   : {len(adv_list)} text entries")
print()
print("Directory layout:")
for p in sorted(WORKDIR.rglob("*"))[:20]:
    rel = p.relative_to(WORKDIR)
    size = f"({p.stat().st_size // 1024} KB)" if p.is_file() else ""
    print(f"  {rel}  {size}")


=== Synthetic wake-word mini-dataset ===
Wake word      : 'hey mycroft'
Positive WAVs  : 20
Total size     : 1156 KB
Avg per sample : 57 KB
Voices used    : 5
Adversarials   : 10 text entries

Directory layout:
  hey_mycroft_adversarials.txt  (0 KB)
  metadata.csv  (0 KB)
  positives  
  positives/pos_0000.wav  (60 KB)
  positives/pos_0001.wav  (57 KB)
  positives/pos_0002.wav  (48 KB)
  positives/pos_0003.wav  (61 KB)
  positives/pos_0004.wav  (61 KB)
  positives/pos_0005.wav  (60 KB)
  positives/pos_0006.wav  (57 KB)
  positives/pos_0007.wav  (48 KB)
  positives/pos_0008.wav  (61 KB)
  positives/pos_0009.wav  (61 KB)
  positives/pos_0010.wav  (60 KB)
  positives/pos_0011.wav  (57 KB)
  positives/pos_0012.wav  (48 KB)
  positives/pos_0013.wav  (61 KB)
  positives/pos_0014.wav  (61 KB)
  positives/pos_0015.wav  (60 KB)
  positives/pos_0016.wav  (57 KB)


## 6 · Next steps

To train a wake-word model from this dataset:

```bash
pip install ww_trainer[datagen]
python -m ww_trainer train \
    --wake-word "hey mycroft" \
    --dataset-dir /path/to/positives \
    --tier micro
```

Or open `kaggle_quickstart_ww.ipynb` for the full GPU-accelerated training flow.

Full augmentation (voice conversion, AudioSet negatives) is covered in
`tts2ww_full_pipeline.ipynb`.


In [7]:
# Cleanup
import shutil
shutil.rmtree(WORKDIR, ignore_errors=True)
print("Workdir cleaned up.")


Workdir cleaned up.
